# Hotel Reviews SQL Practice

Querying `data/hotel_reviews.db` (DuckDB).  
Tables: `HOTEL`, `AGODA_REVIEW`, `GOOGLEMAPS_REVIEW`  
View: `REVIEW_DATA` (merged, both sources)

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/hotel_reviews.db', read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run SQL and return a DataFrame."""
    return con.execute(sql).df()

print('Connected.')

Connected.


---
## 1. Schema overview

In [3]:
# All tables and views in the database
q("SHOW TABLES")

,name
0,AGODA_REVIEW
1,GOOGLEMAPS_REVIEW
2,HOTEL
3,REVIEW_DATA
4,REVIEW_EMBEDDINGS
5,REVIEW_TEXT_PROCESSED
6,REVIEW_TOPICS
7,TOPIC_LABELS


In [9]:
q("""SELECT review_text, processed_review_text
  FROM REVIEWS_DATA, REVIEW_DATA""")

CatalogException: Catalog Error: Table with name REVIEWS_DATA does not exist!
Did you mean "REVIEW_DATA"?

LINE 2:   FROM REVIEWS_DATA, REVIEW_DATA
               ^

In [2]:
# Row counts for everything
q("""
SELECT 'HOTEL'             AS name, COUNT(*) AS rows FROM HOTEL
UNION ALL
SELECT 'AGODA_REVIEW',              COUNT(*)         FROM AGODA_REVIEW
UNION ALL
SELECT 'GOOGLEMAPS_REVIEW',         COUNT(*)         FROM GOOGLEMAPS_REVIEW
UNION ALL
SELECT 'REVIEW_DATA (view)',        COUNT(*)         FROM REVIEW_DATA
""")

,name,rows
0,HOTEL,8574
1,AGODA_REVIEW,125347
2,GOOGLEMAPS_REVIEW,125981
3,REVIEW_DATA (view),251328


In [ ]:
# Columns of each table
for table in ['HOTEL', 'AGODA_REVIEW', 'GOOGLEMAPS_REVIEW']:
    print(f'\n=== {table} ===')
    display(q(f"DESCRIBE {table}"))

---
## 2. HOTEL

In [ ]:
# Preview
q("SELECT * FROM HOTEL LIMIT 5")

In [ ]:
# Hotels by city â€” how many in each city?
q("""
SELECT city, COUNT(*) AS hotel_count
FROM HOTEL
GROUP BY city
ORDER BY hotel_count DESC
LIMIT 10
""")

In [ ]:
# Star rating distribution
q("""
SELECT star_rating, COUNT(*) AS count
FROM HOTEL
GROUP BY star_rating
ORDER BY star_rating
""")

In [ ]:
# Hotels closest to the coast (top 10)
q("""
SELECT hotel_id, hotel_name, city, star_rating,
       ROUND(distance2coastline, 2) AS km_to_coast
FROM HOTEL
WHERE distance2coastline IS NOT NULL
ORDER BY distance2coastline
LIMIT 10
""")

---
## 3. AGODA_REVIEW

In [ ]:
# Preview â€” text + key metadata
q("""
SELECT review_id, hotel_id, language, score,
       stay_year, stay_month, reviewer_nationality,
       LEFT(review_text, 100) AS review_snippet
FROM AGODA_REVIEW
LIMIT 5
""")

In [ ]:
# Language breakdown
q("""
SELECT language, COUNT(*) AS count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM AGODA_REVIEW
GROUP BY language
ORDER BY count DESC
""")

In [ ]:
# Score distribution (rounded to nearest 0.5)
q("""
SELECT ROUND(score * 2) / 2 AS score_bucket,
       COUNT(*) AS count
FROM AGODA_REVIEW
WHERE score IS NOT NULL
GROUP BY score_bucket
ORDER BY score_bucket
""")

In [ ]:
# Top reviewer nationalities
q("""
SELECT reviewer_nationality, COUNT(*) AS reviews
FROM AGODA_REVIEW
WHERE reviewer_nationality IS NOT NULL
GROUP BY reviewer_nationality
ORDER BY reviews DESC
LIMIT 10
""")

In [ ]:
# Reviews per month (trend)
q("""
SELECT stay_year, stay_month, COUNT(*) AS reviews
FROM AGODA_REVIEW
WHERE stay_year IS NOT NULL
GROUP BY stay_year, stay_month
ORDER BY stay_year, stay_month
""")

---
## 4. GOOGLEMAPS_REVIEW

In [ ]:
# Preview
q("""
SELECT review_id, hotel_id, language, rating,
       is_local_guide, reviewer_reviews,
       tag_room, tag_service, tag_location,
       LEFT(review_text, 100) AS review_snippet
FROM GOOGLEMAPS_REVIEW
LIMIT 5
""")

In [ ]:
# Rating distribution (1â€“5 scale)
q("""
SELECT rating, COUNT(*) AS count
FROM GOOGLEMAPS_REVIEW
WHERE rating IS NOT NULL
GROUP BY rating
ORDER BY rating
""")

In [ ]:
# How many reviews have structured Google tags populated?
q("""
SELECT
    COUNT(*) FILTER (WHERE tag_room     IS NOT NULL) AS has_tag_room,
    COUNT(*) FILTER (WHERE tag_service  IS NOT NULL) AS has_tag_service,
    COUNT(*) FILTER (WHERE tag_location IS NOT NULL) AS has_tag_location,
    COUNT(*) FILTER (WHERE tag_food_drink IS NOT NULL) AS has_tag_food,
    COUNT(*) AS total
FROM GOOGLEMAPS_REVIEW
""")

In [ ]:
# Most common tag_room values
q("""
SELECT tag_room, COUNT(*) AS count
FROM GOOGLEMAPS_REVIEW
WHERE tag_room IS NOT NULL AND tag_room != ''
GROUP BY tag_room
ORDER BY count DESC
LIMIT 15
""")

---
## 5. REVIEW_DATA â€” unified view

In [ ]:
# Reviews per source and language
q("""
SELECT source, language, COUNT(*) AS reviews
FROM REVIEW_DATA
GROUP BY source, language
ORDER BY source, reviews DESC
""")

In [ ]:
# Average normalized rating per source
q("""
SELECT source,
       ROUND(AVG(rating_normalized), 3) AS avg_rating,
       ROUND(MIN(rating_normalized), 3) AS min_rating,
       ROUND(MAX(rating_normalized), 3) AS max_rating
FROM REVIEW_DATA
WHERE rating_normalized IS NOT NULL
GROUP BY source
""")

In [ ]:
# Top 10 most-reviewed hotels (across both sources)
q("""
SELECT hotel_id, hotel_name, city, star_rating,
       COUNT(*) AS total_reviews,
       COUNT(*) FILTER (WHERE source = 'agoda')       AS agoda_reviews,
       COUNT(*) FILTER (WHERE source = 'googlemaps')  AS gmap_reviews,
       ROUND(AVG(rating_normalized), 2)               AS avg_rating
FROM REVIEW_DATA
GROUP BY hotel_id, hotel_name, city, star_rating
ORDER BY total_reviews DESC
LIMIT 10
""")

In [ ]:
# English reviews only â€” ready for LLM labeling
q("""
SELECT source, COUNT(*) AS unlabeled_en
FROM REVIEW_DATA
WHERE language = 'en'
  AND labeled_at IS NULL
GROUP BY source
""")

In [ ]:
# Sample of English reviews that need labeling (what label.py will process)
q("""
SELECT source, review_id, hotel_name, rating_normalized,
       LEFT(review_text, 120) AS review_snippet
FROM REVIEW_DATA
WHERE language = 'en'
  AND labeled_at IS NULL
  AND review_text IS NOT NULL
  AND TRIM(review_text) != ''
ORDER BY RANDOM()
LIMIT 10
""")

---
## 6. Joining HOTEL + reviews

In [ ]:
# Average Agoda score per hotel â€” top 10 best rated (min 20 reviews)
q("""
SELECT h.hotel_id, h.hotel_name, h.city, h.star_rating,
       ROUND(h.distance2coastline, 1) AS km_to_coast,
       COUNT(a.review_id)             AS review_count,
       ROUND(AVG(a.score), 2)         AS avg_score
FROM HOTEL h
JOIN AGODA_REVIEW a ON a.hotel_id = h.hotel_id
GROUP BY h.hotel_id, h.hotel_name, h.city, h.star_rating, h.distance2coastline
HAVING review_count >= 20
ORDER BY avg_score DESC
LIMIT 10
""")

In [ ]:
# Hotels that appear in BOTH sources
q("""
SELECT h.hotel_id, h.hotel_name, h.city,
       COUNT(DISTINCT a.review_id) AS agoda_reviews,
       COUNT(DISTINCT g.review_id) AS gmap_reviews
FROM HOTEL h
JOIN AGODA_REVIEW     a ON a.hotel_id = h.hotel_id
JOIN GOOGLEMAPS_REVIEW g ON g.hotel_id = h.hotel_id
GROUP BY h.hotel_id, h.hotel_name, h.city
ORDER BY agoda_reviews + gmap_reviews DESC
LIMIT 10
""")

In [ ]:
# Coastal hotels (< 5 km) with their review counts
q("""
SELECT h.hotel_name, h.city, h.star_rating,
       ROUND(h.distance2coastline, 2) AS km_to_coast,
       COUNT(r.review_id)             AS reviews
FROM HOTEL h
JOIN REVIEW_DATA r ON r.hotel_id = h.hotel_id
WHERE h.distance2coastline < 5
  AND r.language = 'en'
GROUP BY h.hotel_id, h.hotel_name, h.city, h.star_rating, h.distance2coastline
ORDER BY h.distance2coastline
LIMIT 15
""")

---
## 7. Queries you'll use after LLM labeling

These cells use the `asp_*` columns which are `NULL` now.  
They'll return real data once `scripts/label.py` runs.

In [ ]:
# Labeling progress
q("""
SELECT source,
       COUNT(*) FILTER (WHERE labeled_at IS NOT NULL) AS labeled,
       COUNT(*) FILTER (WHERE labeled_at IS NULL)     AS unlabeled,
       COUNT(*)                                        AS total
FROM REVIEW_DATA
WHERE language = 'en'
GROUP BY source
""")

In [ ]:
# Sentiment distribution per aspect (run after labeling)
q("""
SELECT
    aspect,
    sentiment,
    COUNT(*) AS count
FROM (
    SELECT 'room'       AS aspect, asp_room       AS sentiment FROM REVIEW_DATA WHERE language = 'en'
    UNION ALL
    SELECT 'service',              asp_service              FROM REVIEW_DATA WHERE language = 'en'
    UNION ALL
    SELECT 'location',             asp_location             FROM REVIEW_DATA WHERE language = 'en'
    UNION ALL
    SELECT 'food_drink',           asp_food_drink           FROM REVIEW_DATA WHERE language = 'en'
    UNION ALL
    SELECT 'value',                asp_value                FROM REVIEW_DATA WHERE language = 'en'
    UNION ALL
    SELECT 'cleanliness',          asp_cleanliness          FROM REVIEW_DATA WHERE language = 'en'
)
WHERE sentiment IS NOT NULL
GROUP BY aspect, sentiment
ORDER BY aspect, sentiment
""")

In [ ]:
# RAG filter example: hotels with positive location + rating >= 4
q("""
SELECT hotel_name, city, star_rating,
       COUNT(*) AS matching_reviews
FROM REVIEW_DATA
WHERE asp_location      = 'positive'
  AND rating_normalized >= 4
  AND language           = 'en'
GROUP BY hotel_id, hotel_name, city, star_rating
ORDER BY matching_reviews DESC
LIMIT 10
""")